In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("set")

set


In [2]:
!pip install -q -U transformers accelerate

In [3]:
import os, torch, transformers
d = '/kaggle/input/datasets/arnobsaha1/weightgait-pilot'
print("dataset files:", len(os.listdir(d)))
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("transformers:", transformers.__version__)
p = '/kaggle/working/results/gemma_runs.jsonl'
print("results file exists:", os.path.exists(p), "| rows:", sum(1 for _ in open(p)) if os.path.exists(p) else 0)

dataset files: 105
GPUs: 2 ['Tesla T4', 'Tesla T4']
transformers: 5.19.0
results file exists: True | rows: 384


In [ ]:
import sys, json
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')
import run_gemma, gemma_backend as gb
from samples import DEV_INSTANCES
gb.MAX_SOFT_TOKENS = 140
REPS = ["raw_video", "skeleton", "overlay", "montage"]
SKIP = {582}   # 21 images exceeds GPU memory with the long P3 prompt
run_gemma.run([i for i in DEV_INSTANCES if i not in SKIP], REPS + ["blind"], ["P3_guided_balanced"], [1])
_ = run_gemma.summarize()

384 calls on disk, 96 to run


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

[1/96] P3_guided_balanced 1fps raw_video 703: gt=2 pred=1 (WRONG, 5 img, 26s)
[2/96] P3_guided_balanced 1fps raw_video 770: gt=2 pred=0 (WRONG, 4 img, 15s)
[3/96] P3_guided_balanced 1fps raw_video 957: gt=2 pred=0 (WRONG, 8 img, 20s)
[4/96] P3_guided_balanced 1fps raw_video 1247: gt=2 pred=0 (WRONG, 4 img, 19s)
[5/96] P3_guided_balanced 1fps raw_video 1369: gt=2 pred=1 (WRONG, 5 img, 17s)
[6/96] P3_guided_balanced 1fps raw_video 1607: gt=2 pred=1 (WRONG, 5 img, 19s)
[7/96] P3_guided_balanced 1fps raw_video 1678: gt=2 pred=1 (WRONG, 5 img, 15s)
[8/96] P3_guided_balanced 1fps raw_video 1799: gt=2 pred=1 (WRONG, 5 img, 17s)
[9/96] P3_guided_balanced 1fps skeleton 427: gt=0 pred=1 (WRONG, 3 img, 15s)
[10/96] P3_guided_balanced 1fps skeleton 548: gt=0 pred=2 (WRONG, 15 img, 26s)
[11/96] P3_guided_balanced 1fps skeleton 675: gt=0 pred=0 (OK, 5 img, 20s)
[12/96] P3_guided_balanced 1fps skeleton 739: gt=0 pred=1 (WRONG, 4 img, 16s)
[13/96] P3_guided_balanced 1fps skeleton 919: gt=0 pred=0 (OK,

In [ ]:
import sys; sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')
import run_gemma
run_gemma.summarize()

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, json
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')
import run_gemma, gemma_backend as gb
from samples import DEV_INSTANCES
gb.MAX_SOFT_TOKENS = 140
REPS = ["raw_video", "skeleton", "overlay", "montage"]
run_gemma.run(DEV_INSTANCES, REPS + ["blind"], ["P3_guided_balanced"], [1])
run_gemma.summarize()

In [ ]:
import sys, json
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')
from pathlib import Path
import run_gemma, gemma_backend as gb
from samples import DEV_INSTANCES
from prompts import PROMPTS
print(list(PROMPTS))
print("SOURCES in P3:", "SOURCES" in PROMPTS["P3_guided_balanced"])

gb.MAX_SOFT_TOKENS = 140
Path("/kaggle/working/results").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/results/run_config.json").write_text(json.dumps({
    "model": gb.MODEL_ID, "seed": gb.SEED, "decoding": "greedy (do_sample=False)",
    "dtype": gb.DTYPE, "max_soft_tokens": gb.MAX_SOFT_TOKENS, "fps": 1, "mode": "window",
}, indent=1))

REPS = ["raw_video", "skeleton", "overlay", "montage"]
PROMPT_NAMES = ["P0_rubric", "P1_reasoning_first", "P2_guided_per_leg", "P3_guided_balanced"]
run_gemma.run(DEV_INSTANCES, REPS + ["blind"], PROMPT_NAMES, [1])
run_gemma.summarize()

In [ ]:
!ls /kaggle/input/datasets/arnobsaha1/weightgait-pilot | head -40
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print(transformers.__version__)

In [ ]:
!ls /kaggle/input/datasets/arnobsaha1/weightgait-pilot | grep -E "windows|samples|prompts|run_gemma"

In [ ]:
import sys, json
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')
import run_gemma, gemma_backend as gb
from samples import DEV_INSTANCES, CLASS_OF
from prompts import PROMPTS

windows = json.loads((run_gemma.DATA_DIR / 'windows.json').read_text())
cuts = json.loads((run_gemma.DATA_DIR / 'ankle_cuts.json').read_text())
i = DEV_INSTANCES[0]
images = run_gemma.load_inputs(i, 'raw_video', 2, 'window', windows, cuts)
print(i, 'true class', CLASS_OF[i], 'images', len(images))
print(gb.generate_fn(images, PROMPTS['P1_reasoning_first']))

In [ ]:
import time
m, p = gb._load()
print("loaded as:", m.config._name_or_path)
print("dtype    :", next(m.parameters()).dtype)
print("devices  :", sorted(set(str(d) for d in m.hf_device_map.values())))

for rep, fps in [("raw_video", 2), ("raw_video", 3), ("montage", 2)]:
    imgs = run_gemma.load_inputs(i, rep, fps, "window", windows, cuts)
    t0 = time.time()
    gb.generate_fn(imgs, PROMPTS["P1_reasoning_first"])
    print(f"{rep} {fps}fps: {len(imgs)} images, {time.time() - t0:.1f}s")

In [ ]:
import sys, json, time, importlib
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')
print("new backend uploaded:", "MAX_SOFT_TOKENS" in open('/kaggle/input/datasets/arnobsaha1/weightgait-pilot/gemma_backend.py').read())

import run_gemma, gemma_backend as gb
from samples import DEV_INSTANCES, CLASS_OF
from prompts import PROMPTS

windows = json.loads((run_gemma.DATA_DIR / 'windows.json').read_text())
cuts = json.loads((run_gemma.DATA_DIR / 'ankle_cuts.json').read_text())
i = DEV_INSTANCES[0]

imgs1 = run_gemma.load_inputs(i, "raw_video", 1, "window", windows, cuts)
print(len(imgs1), "images at 1 fps")
for budget in (70, 140, 280):
    gb.MAX_SOFT_TOKENS = budget
    n = gb.count_input_tokens(imgs1, PROMPTS["P1_reasoning_first"])
    t0 = time.time()
    out = gb.generate_fn(imgs1, PROMPTS["P1_reasoning_first"])
    print(f"budget {budget}: {n} input tokens, {time.time() - t0:.1f}s -> {out[:90]}")

In [ ]:
import json
from pathlib import Path

gb.MAX_SOFT_TOKENS = 140
Path("/kaggle/working/results").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/results/run_config.json").write_text(json.dumps({
    "model": gb.MODEL_ID, "seed": gb.SEED, "decoding": "greedy (do_sample=False)",
    "dtype": gb.DTYPE, "max_soft_tokens": gb.MAX_SOFT_TOKENS, "fps": 1, "mode": "window",
}, indent=1))

REPS = ["raw_video", "skeleton", "overlay", "montage"]
PROMPT_NAMES = ["P0_rubric", "P1_reasoning_first", "P2_per_leg", "P3_balanced_prior"]
run_gemma.run(DEV_INSTANCES, REPS + ["blind"], PROMPT_NAMES, [1])   # Stage A
run_gemma.summarize()

In [ ]:
import os
p = "/kaggle/working/results/gemma_runs.jsonl"
print("results file exists:", os.path.exists(p))
if os.path.exists(p):
    print("rows:", sum(1 for _ in open(p)))
import transformers
print("transformers", transformers.__version__)
!nvidia-smi --query-gpu=index,memory.used --format=csv

In [ ]:
import json, shutil
p = "/kaggle/working/results/gemma_runs.jsonl"
shutil.copy(p, "/kaggle/working/results/gemma_runs_backup.jsonl")
rows = [json.loads(l) for l in open(p)]
keep = [r for r in rows if r["prompt"] in ("P0_rubric", "P1_reasoning_first")]
open(p, "w").write("".join(json.dumps(r) + "\n" for r in keep))
print(len(rows), "->", len(keep), "rows")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys, json
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')
from pathlib import Path
import run_gemma, gemma_backend as gb
from samples import DEV_INSTANCES
from prompts import PROMPTS
print(list(PROMPTS))
print("integer rule in P2/P3:", all("integer 0, 1 or 2" in PROMPTS[k] for k in ("P2_guided_per_leg", "P3_guided_balanced")))
print("SOURCES in P3:", "SOURCES" in PROMPTS["P3_guided_balanced"])

gb.MAX_SOFT_TOKENS = 140
Path("/kaggle/working/results").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/results/run_config.json").write_text(json.dumps({
    "model": gb.MODEL_ID, "seed": gb.SEED, "decoding": "greedy (do_sample=False)",
    "dtype": gb.DTYPE, "max_soft_tokens": gb.MAX_SOFT_TOKENS, "fps": 1, "mode": "window",
}, indent=1))

REPS = ["raw_video", "skeleton", "overlay", "montage"]
PROMPT_NAMES = ["P2_guided_per_leg", "P3_guided_balanced"]
run_gemma.run(DEV_INSTANCES, REPS + ["blind"], PROMPT_NAMES, [1])
run_gemma.summarize()

In [ ]:
import json, collections
c = collections.Counter()
for line in open("/kaggle/working/results/gemma_runs.jsonl"):
    r = json.loads(line); c[(r["prompt"], r["rep"])] += 1
for k, v in sorted(c.items()): print(k, v)